# Error Analysis on Sample Images
This notebook runs local error analysis using the saved models in `Models/` and the images in the `samples/` directory. 
It helps us visualize how both the Baseline CNN and MobileNetV3 perform on unseen data, specifically highlighting any misclassifications.

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

# Add project root to python path so we can import src
project_root = Path(os.getcwd()).parent
sys.path.append(str(project_root))

from src.model import load_model
from src.inference import predict

## 1. Load Models
Load the pre-trained weights from the `Models/` directory.

In [ ]:
print("Loading CNN...")
cnn_model = load_model("cnn")
print("Loading MobileNetV3...")
mob_model = load_model("mobilenetv3")

## 2. Load and Predict on Samples
We will iterate through the `samples/` directory, extract the true label from the filename, and run predictions.

In [ ]:
samples_dir = project_root / "samples"
sample_files = [f for f in samples_dir.iterdir() if f.suffix.lower() in ['.png', '.jpg', '.jpeg']]

results = []

for file_path in sample_files:
    # Determine true label from filename (e.g. image1_positive.png)
    filename = file_path.name.lower()
    if 'positive' in filename or 'postive' in filename:
        true_label = 'fractured'
    elif 'negative' in filename:
        true_label = 'not_fractured'
    else:
        continue # Skip if we can't determine ground truth
        
    # Get predictions
    cnn_res = predict(cnn_model, file_path, model_name="cnn")
    mob_res = predict(mob_model, file_path, model_name="mobilenetv3")
    
    results.append({
        "path": file_path,
        "true": true_label,
        "cnn_pred": cnn_res["prediction"],
        "cnn_conf": cnn_res["confidence"],
        "mob_pred": mob_res["prediction"],
        "mob_conf": mob_res["confidence"]
    })

print(f"Processed {len(results)} labeled sample images.")

## 3. Visualize Results & Errors
Let's plot the images and compare the predictions from both models. Incorrect predictions will be highlighted in **red**.

In [ ]:
fig, axes = plt.subplots(len(results), 2, figsize=(10, 4 * len(results)))
if len(results) == 1:
    axes = [axes]

for i, res in enumerate(results):
    img = Image.open(res["path"]).convert("RGB")
    
    # CNN Plot
    ax_cnn = axes[i][0]
    ax_cnn.imshow(img)
    ax_cnn.axis("off")
    cnn_color = "green" if res["cnn_pred"] == res["true"] else "red"
    ax_cnn.set_title(f"CNN (True: {res['true']})\nPred: {res['cnn_pred']} ({res['cnn_conf']:.2f})", color=cnn_color, fontsize=10)
    
    # MobileNetV3 Plot
    ax_mob = axes[i][1]
    ax_mob.imshow(img)
    ax_mob.axis("off")
    mob_color = "green" if res["mob_pred"] == res["true"] else "red"
    ax_mob.set_title(f"MobileNetV3 (True: {res['true']})\nPred: {res['mob_pred']} ({res['mob_conf']:.2f})", color=mob_color, fontsize=10)

plt.tight_layout()
plt.show()